# 02 — Benchmarking Analysis

Core analysis: U.S. vs. peer average, z-scores among high-income countries, percentile rankings, and the "income group impersonation" test.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
from scipy import stats
from helpers import (
    load_data, get_country, get_countries, get_peer_average,
    compute_pct_difference, assign_grade, normalize_score,
    PEER_COUNTRIES, PEER_COUNTRIES_NO_US, AGGREGATES,
    FOCUS_VARIABLES, FOCUS_COLS, VAR_LABELS, VAR_LOWER_IS_BETTER,
    COUNTRY_NAMES, COLORS
)

df = load_data()
usa = get_country(df, 'USA')
SNAPSHOT_YEAR = 2010

## 1. U.S. vs. Peer Average — Development Scorecard (2010)

In [ ]:
usa_row = usa[usa['Year'] == SNAPSHOT_YEAR].iloc[0]

peers_year = df[
    (df['Country.Code'].isin(PEER_COUNTRIES_NO_US)) &
    (df['Year'] == SNAPSHOT_YEAR)
]
peer_avg = peers_year[FOCUS_COLS].mean()

scorecard = []
for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    usa_val = usa_row[col]
    peer_val = peer_avg[col]
    if pd.isna(usa_val) or pd.isna(peer_val):
        continue
    
    pct_diff = compute_pct_difference(usa_val, peer_val)
    grade = assign_grade(pct_diff, lower_is_better)
    
    scorecard.append({
        'Indicator': label,
        'MDG Goal': goal,
        'USA': round(usa_val, 2),
        'Peer Average': round(peer_val, 2),
        '% Difference': round(pct_diff, 1),
        'Lower is Better': lower_is_better,
        'Grade': grade,
    })

scorecard_df = pd.DataFrame(scorecard)
scorecard_df.sort_values('Grade')

## 2. Z-Scores — U.S. Position Among All High-Income Countries

In [ ]:
# Get the High Income aggregate to identify HIC countries
# We'll use individual countries that are typically classified as high-income
# For simplicity, we'll use all countries and compare against the HIC aggregate

# List of well-known high-income country codes
HIGH_INCOME_CODES = [
    'USA', 'GBR', 'DEU', 'JPN', 'CAN', 'FRA', 'AUS', 'NLD', 'SWE', 'NOR',
    'ITA', 'ESP', 'KOR', 'CHE', 'AUT', 'BEL', 'DNK', 'FIN', 'IRL', 'NZL',
    'PRT', 'SGP', 'ISR', 'CZE', 'GRC', 'SVN', 'EST', 'LTU', 'LVA', 'SVK',
    'HRV', 'POL', 'HUN', 'CHL', 'URY', 'SAU', 'ARE', 'KWT', 'QAT', 'BHR',
    'OMN', 'ISL', 'LUX', 'MLT', 'CYP', 'BRN',
]

hic_data = df[
    (df['Country.Code'].isin(HIGH_INCOME_CODES)) &
    (df['Year'] == SNAPSHOT_YEAR)
]

zscore_results = []
for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    values = hic_data[col].dropna()
    usa_val = usa_row[col]
    
    if pd.isna(usa_val) or len(values) < 5:
        continue
    
    mean_val = values.mean()
    std_val = values.std()
    
    if std_val == 0:
        continue
    
    z = (usa_val - mean_val) / std_val
    
    # Percentile rank among HIC
    if lower_is_better:
        # For lower-is-better, rank ascending (lower value = better rank)
        pctile = stats.percentileofscore(values, usa_val, kind='rank')
        pctile_quality = 100 - pctile  # Invert so higher = better
    else:
        pctile = stats.percentileofscore(values, usa_val, kind='rank')
        pctile_quality = pctile
    
    zscore_results.append({
        'Indicator': label,
        'USA Value': round(usa_val, 2),
        'HIC Mean': round(mean_val, 2),
        'HIC Std': round(std_val, 2),
        'Z-Score': round(z, 2),
        'Percentile (quality)': round(pctile_quality, 1),
        'N Countries': len(values),
    })

zscore_df = pd.DataFrame(zscore_results)
zscore_df.sort_values('Percentile (quality)')

## 3. Income Group Impersonation

For each indicator, which income group does the U.S. most resemble?

In [ ]:
agg_codes = ['HIC', 'UMC', 'LMC', 'WLD']
agg_labels = {k: AGGREGATES[k] for k in agg_codes}

agg_data = df[
    (df['Country.Code'].isin(agg_codes)) &
    (df['Year'] == SNAPSHOT_YEAR)
]

impersonation = []
for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    usa_val = usa_row[col]
    if pd.isna(usa_val):
        continue
    
    row = {'Indicator': label, 'USA': round(usa_val, 2)}
    
    closest_group = None
    closest_dist = float('inf')
    
    for code in agg_codes:
        agg_row = agg_data[agg_data['Country.Code'] == code]
        if len(agg_row) > 0:
            agg_val = agg_row.iloc[0][col]
            if pd.notna(agg_val):
                row[agg_labels[code]] = round(agg_val, 2)
                dist = abs(usa_val - agg_val)
                if dist < closest_dist:
                    closest_dist = dist
                    closest_group = agg_labels[code]
    
    # Check if USA is an outlier (above HIC on a lower-is-better metric)
    hic_val = agg_data[agg_data['Country.Code'] == 'HIC'].iloc[0][col] if 'HIC' in agg_data['Country.Code'].values else None
    if hic_val is not None and pd.notna(hic_val):
        if lower_is_better and usa_val > hic_val * 1.5:
            closest_group = f'Outlier (above {agg_labels["HIC"]})'
        elif not lower_is_better and usa_val < hic_val * 0.5:
            closest_group = f'Outlier (below {agg_labels["HIC"]})'
    
    row['Acts Like'] = closest_group
    impersonation.append(row)

imp_df = pd.DataFrame(impersonation)
imp_df

## 4. Individual Peer Country Comparison (2010)

Where does the U.S. rank among the 6 peer countries on each indicator?

In [ ]:
peers_snapshot = df[
    (df['Country.Code'].isin(PEER_COUNTRIES)) &
    (df['Year'] == SNAPSHOT_YEAR)
].copy()

ranking_results = []
for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    subset = peers_snapshot[['Country.Code', col]].dropna()
    if len(subset) < 3:
        continue
    
    # Rank: 1 = best
    if lower_is_better:
        subset['Rank'] = subset[col].rank(ascending=True).astype(int)
    else:
        subset['Rank'] = subset[col].rank(ascending=False).astype(int)
    
    usa_rank = subset[subset['Country.Code'] == 'USA']['Rank']
    if len(usa_rank) > 0:
        ranking_results.append({
            'Indicator': label,
            'USA Rank': usa_rank.iloc[0],
            'Out Of': len(subset),
            'USA Value': round(subset[subset['Country.Code'] == 'USA'][col].iloc[0], 2),
            'Best Value': round(subset[col].min() if lower_is_better else subset[col].max(), 2),
            'Best Country': subset.loc[
                subset[col].idxmin() if lower_is_better else subset[col].idxmax(),
                'Country.Code'
            ],
        })

rank_df = pd.DataFrame(ranking_results)
rank_df.sort_values('USA Rank', ascending=False)

## 5. Save Processed Data for Visualization

In [ ]:
# Save key dataframes for use in charts.py
scorecard_df.to_csv('../analysis/scorecard.csv', index=False)
zscore_df.to_csv('../analysis/zscores.csv', index=False)
imp_df.to_csv('../analysis/income_impersonation.csv', index=False)
rank_df.to_csv('../analysis/peer_rankings.csv', index=False)

print('Saved: scorecard.csv, zscores.csv, income_impersonation.csv, peer_rankings.csv')